## A.2. DDI and true-system intervention response

Compare DDI with the mean squared difference between control and edge-removal target trajectories in a multi-delay Lorenz system. Matched histories and dynamical-noise realizations are used for each true edge-delay pair.

Existing results are reused; missing observations or model results are computed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from scipy.stats import spearmanr
from matplotlib.ticker import MaxNLocator
%run ./ESN.ipynb
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.size'] = 16
plt.rcParams['axes.titlesize'] = 18
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['xtick.labelsize'] = 14
plt.rcParams['ytick.labelsize'] = 14
plt.rcParams['legend.fontsize'] = 12
N_x = 4
N_washout = 100
N_train = 10000
N_tstart = 10000
N_test = 2000
window = 6
window_length = 100
test_window = 6
lag_list = np.arange(test_window) * window_length
N_net = 20
N_rep = 30
N_evo = 5
sigma_dyn = 1e-06
sigma_obs = 1e-06
c = 0.3
dt = 0.01
density = 0.2
NETWORK_SEED = 99
DATA_SEED = 199
OFFSET_SEED = 299
TRUE_NOISE_SEED = 399
RESERVOIR_SEED = 499
TRUE_RESULT_PATH = os.path.join('result', 'A.2', 'A.2_true.pkl')
RESULT_PATH = os.path.join('result', 'A.2', 'A.2.pkl')
FIGURE_PATH = os.path.join('result', 'figure', 'A.2.png')
os.makedirs(os.path.dirname(TRUE_RESULT_PATH), exist_ok=True)
os.makedirs(os.path.dirname(FIGURE_PATH), exist_ok=True)

def read_mean_squared_response(data, level):
    for prefix in ('R_MS', 'delta_U', 'DDI_true'):
        key = f'{prefix}_{level}'
        if key in data:
            return np.asarray(data[key], dtype=float)
    raise KeyError(f'Missing mean squared intervention response for level: {level}')


### Part 1. Load or compute true-system responses

Reuse the saved trajectory, normalization and initial histories. Missing responses are computed by five-step numerical integration. The response is denoted by $R_{\mathrm{MS}}$.


In [ ]:
def lorenz_rhs_6delay_with_noise(states, h, coupling_strength, A_list, noise):
    """Lorenz derivative with an explicit noise vector for paired rollouts."""
    current = states[0]
    derivative = np.zeros_like(current)
    for node in range(N_x):
        x_value = current[3 * node + 0]
        y_value = current[3 * node + 1]
        z_value = current[3 * node + 2]
        coupling = 0.0
        for lag_idx in range(test_window):
            coupling += np.sum(A_list[lag_idx][node] * (states[lag_idx, 1::3] - y_value))
        derivative[3 * node + 0] = -10.0 * (x_value - y_value + coupling_strength * coupling) + noise[3 * node + 0]
        derivative[3 * node + 1] = 28.0 * (1.0 + h[node]) * x_value - y_value - x_value * z_value + noise[3 * node + 1]
        derivative[3 * node + 2] = -8.0 / 3.0 * z_value + x_value * y_value + noise[3 * node + 2]
    return derivative

def rk4_step_from_reference(current, reference, absolute_index, h, coupling_strength, A_list, noise_stages):
    """One RK4 step using the same delay convention as RK4_6delay."""
    delayed_start = [reference[absolute_index - lag_idx * window_length] for lag_idx in range(1, test_window)]
    states_1 = np.vstack([current] + delayed_start)
    k1 = lorenz_rhs_6delay_with_noise(states_1, h, coupling_strength, A_list, noise_stages[0])
    delayed_mid = [(reference[absolute_index - lag_idx * window_length] + reference[absolute_index - lag_idx * window_length + 1]) / 2.0 for lag_idx in range(1, test_window)]
    states_2 = np.vstack([current + dt / 2.0 * k1] + delayed_mid)
    k2 = lorenz_rhs_6delay_with_noise(states_2, h, coupling_strength, A_list, noise_stages[1])
    states_3 = np.vstack([current + dt / 2.0 * k2] + delayed_mid)
    k3 = lorenz_rhs_6delay_with_noise(states_3, h, coupling_strength, A_list, noise_stages[2])
    delayed_end = [reference[absolute_index - lag_idx * window_length + 1] for lag_idx in range(1, test_window)]
    states_4 = np.vstack([current + dt * k3] + delayed_end)
    k4 = lorenz_rhs_6delay_with_noise(states_4, h, coupling_strength, A_list, noise_stages[3])
    return current + dt / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)

def generate_lorenz_baseline(n_data, history_length, h, initial_vector, coupling_strength, A_list, sigma, random_state):
    """Generate the raw Lorenz trajectory while retaining its parameters."""
    X_raw = np.zeros((n_data, 3 * N_x))
    X_raw[:history_length] = np.tile(initial_vector, (history_length, 1))
    for absolute_index in range(history_length - 1, n_data - 1):
        noise_stages = random_state.normal(loc=0.0, scale=sigma, size=(4, 3 * N_x))
        X_raw[absolute_index + 1] = rk4_step_from_reference(X_raw[absolute_index], X_raw, absolute_index, h, coupling_strength, A_list, noise_stages)
    return X_raw

def rollout_true_dynamics(reference, start_index, h, coupling_strength, A_list, noise_path):
    """Five-step rollout from a fixed history with a supplied noise path."""
    response = np.zeros((N_evo + 1, reference.shape[1]))
    response[0] = reference[start_index]
    for step in range(N_evo):
        absolute_index = start_index + step
        response[step + 1] = rk4_step_from_reference(response[step], reference, absolute_index, h, coupling_strength, A_list, noise_path[step])
    return response
if os.path.exists(TRUE_RESULT_PATH):
    with open(TRUE_RESULT_PATH, 'rb') as stream:
        true_result = pickle.load(stream)
    globals().update(true_result)
    R_MS_rep = read_mean_squared_response(true_result, 'rep')
    R_MS_net = read_mean_squared_response(true_result, 'net')
    A_list = list(A_tensor)
    max_lag = int(lag_list.max())
else:
    network_rng = np.random.RandomState(NETWORK_SEED)
    A_list = []
    for _ in range(test_window):
        A = (network_rng.rand(N_x, N_x) < density).astype(float)
        np.fill_diagonal(A, 0.0)
        A_list.append(A)
    A_list[3][0, 1] = 1.0
    A_tensor = np.stack(A_list, axis=0)
    true_edge_mask = A_tensor.astype(bool)
    data_rng = np.random.RandomState(DATA_SEED)
    h = 2.0 * (data_rng.rand(N_x) - 0.5) * 0.06
    initial_vector = data_rng.rand(3 * N_x)
    history_length = window * window_length
    n_data = N_tstart + N_test
    X_raw = generate_lorenz_baseline(n_data, history_length, h, initial_vector, c, A_list, sigma_dyn, data_rng)
    q1 = np.percentile(X_raw, 25, interpolation='midpoint', axis=0)
    q2 = np.percentile(X_raw, 50, interpolation='midpoint', axis=0)
    q3 = np.percentile(X_raw, 75, interpolation='midpoint', axis=0)
    iqr = q3 - q1
    if np.any(np.abs(iqr) < 1e-12):
        raise ValueError('At least one Lorenz component has a near-zero interquartile range.')
    X_observed = (X_raw - q2) / iqr
    if sigma_obs != 0:
        X_observed += data_rng.normal(0.0, sigma_obs, size=X_observed.shape)
    max_lag = int(lag_list.max())
    (T, D) = X_observed.shape
    X_segments = np.zeros((T - max_lag, test_window * D))
    for (row, absolute_index) in enumerate(range(max_lag, T)):
        X_segments[row] = np.concatenate([X_observed[absolute_index - lag] for lag in lag_list])
    X_washout_segments = X_segments[:N_washout]
    X_t_segments = X_segments[N_washout:N_washout + N_train - 1]
    Y_t_segments = X_segments[N_washout + 1:N_washout + N_train]
    X_test_segments = X_segments[N_tstart:]
    max_offset = X_test_segments.shape[0] - N_evo - 1
    if max_offset <= N_washout:
        raise ValueError('The test sequence is too short for the requested paired rollouts.')
    offset_rng = np.random.RandomState(OFFSET_SEED)
    offset = offset_rng.randint(N_washout, max_offset, size=(N_net, N_rep))
    R_MS_rep = np.zeros((N_net, N_rep, test_window, N_x, N_x), dtype=float)
    response_true = np.zeros((N_net, N_rep, test_window, N_x, N_x, N_evo + 1), dtype=float)
    completed_true_net = np.zeros(N_net, dtype=bool)
    if os.path.exists(TRUE_RESULT_PATH):
        with open(TRUE_RESULT_PATH, 'rb') as f:
            previous_true = pickle.load(f)
        previous_rep = read_mean_squared_response(previous_true, 'rep')
        previous_shape = previous_rep.shape
        if previous_shape == R_MS_rep.shape:
            R_MS_rep[:] = previous_rep
            response_true[:] = previous_true['response_true']
            completed_true_net[:] = previous_true['completed_true_net']
    for net in range(N_net):
        if completed_true_net[net]:
            continue
        for rep in range(N_rep):
            start_index = max_lag + N_tstart + int(offset[net, rep])
            noise_rng = np.random.RandomState(TRUE_NOISE_SEED + net * N_rep + rep)
            common_noise = noise_rng.normal(0.0, sigma_dyn, size=(N_evo, 4, 3 * N_x))
            control_raw = rollout_true_dynamics(X_raw, start_index, h, c, A_list, common_noise)
            for (lag_idx, effect, cause) in np.argwhere(true_edge_mask):
                intervened_A = [A.copy() for A in A_list]
                intervened_A[lag_idx][effect, cause] = 0.0
                intervention_raw = rollout_true_dynamics(X_raw, start_index, h, c, intervened_A, common_noise)
                control_normalized = (control_raw - q2) / iqr
                intervention_normalized = (intervention_raw - q2) / iqr
                delta = control_normalized[:, 3 * effect] - intervention_normalized[:, 3 * effect]
                response_true[net, rep, lag_idx, effect, cause] = delta
                R_MS_rep[net, rep, lag_idx, effect, cause] = np.sum(delta ** 2) / N_evo
        completed_true_net[net] = True
        R_MS_net = np.mean(R_MS_rep, axis=1)
        true_result = {'A_tensor': A_tensor, 'true_edge_mask': true_edge_mask, 'X_raw': X_raw, 'X_observed': X_observed, 'X_segments': X_segments, 'X_washout_segments': X_washout_segments, 'X_t_segments': X_t_segments, 'Y_t_segments': Y_t_segments, 'X_test_segments': X_test_segments, 'h': h, 'q1': q1, 'q2': q2, 'q3': q3, 'iqr': iqr, 'offset': offset, 'R_MS_rep': R_MS_rep, 'R_MS_net': R_MS_net, 'response_true': response_true, 'completed_true_net': completed_true_net, 'lag_list': lag_list, 'config': {'N_x': N_x, 'N_washout': N_washout, 'N_train': N_train, 'N_tstart': N_tstart, 'N_test': N_test, 'window': window, 'window_length': window_length, 'test_window': test_window, 'N_net': N_net, 'N_rep': N_rep, 'N_evo': N_evo, 'sigma_dyn': sigma_dyn, 'sigma_obs': sigma_obs, 'c': c, 'dt': dt, 'density': density, 'NETWORK_SEED': NETWORK_SEED, 'DATA_SEED': DATA_SEED, 'OFFSET_SEED': OFFSET_SEED, 'TRUE_NOISE_SEED': TRUE_NOISE_SEED}}
        with open(TRUE_RESULT_PATH, 'wb') as f:
            pickle.dump(true_result, f)


### Part 2. Compute DDI and plot

Reuse completed ESN results. Otherwise, train 20 models and average the positive-history DDI scores using the existing signed normalization. Plot ensemble-mean DDI against $R_{\mathrm{MS}}$ for true edges, with delay colors, a through-origin fit and Spearman correlation.


In [ ]:
with open(TRUE_RESULT_PATH, 'rb') as f:
    true_data = pickle.load(f)
if not np.all(true_data['completed_true_net']):
    raise RuntimeError('Part 1 is incomplete; finish all true-dynamics blocks first.')
A_tensor = np.asarray(true_data['A_tensor'], dtype=float)
true_edge_mask = np.asarray(true_data['true_edge_mask'], dtype=bool)
X_washout_segments = np.asarray(true_data['X_washout_segments'])
X_t_segments = np.asarray(true_data['X_t_segments'])
Y_t_segments = np.asarray(true_data['Y_t_segments'])
X_test_segments = np.asarray(true_data['X_test_segments'])
offset = np.asarray(true_data['offset'], dtype=int)
R_MS_rep = read_mean_squared_response(true_data, 'rep')
R_MS_net = read_mean_squared_response(true_data, 'net')
response_true = np.asarray(true_data['response_true'], dtype=float)
lag_list = np.asarray(true_data['lag_list'], dtype=int)
N_dim = 3 * N_x
N_dim_segments = test_window * N_dim
N_units_segments = 100 * N_dim_segments

def calculate_istc_response(esn, X_test, offsets, lag_idx, j_threshold=0.01):
    """Return the ISTC matrix, replicate scores, and response trajectories."""
    DDI_rep = np.zeros((N_rep, N_x, N_x), dtype=float)
    response = np.zeros((N_rep, N_x, N_x, N_evo + 1), dtype=float)
    for rep in range(N_rep):
        current_offset = int(offsets[rep])
        X_test_washout = X_test[current_offset - N_washout:current_offset]
        r0 = esn.open_loop(X_test_washout, np.zeros(esn.n_units))[-1]
        control = esn.evolve(r0, N_evo)
        for effect in range(N_x):
            for cause in range(N_x):
                if effect == cause:
                    continue
                target_effect = 3 * effect
                target_cause = 3 * cause + N_dim * lag_idx + 1
                intervention = esn.evolve_edge_removal(r0, target_cause, target_effect, N_evo)
                delta = control[:, target_effect] - intervention[:, target_effect]
                response[rep, effect, cause] = delta
                cause_mean = np.mean(control[:, target_cause])
                if np.abs(cause_mean) < j_threshold:
                    DDI_rep[rep, effect, cause] = -1.0
                else:
                    DDI_rep[rep, effect, cause] = np.sum(delta ** 2) / N_evo / cause_mean
    DDI = np.zeros((N_x, N_x), dtype=float)
    for effect in range(N_x):
        for cause in range(N_x):
            if effect == cause:
                continue
            positive_values = DDI_rep[:, effect, cause]
            positive_values = positive_values[positive_values > 0]
            if positive_values.size > 0:
                DDI[effect, cause] = np.mean(positive_values)
    return (DDI, DDI_rep, response)
DDI_ISTC_net = np.full((N_net, test_window, N_x, N_x), np.nan)
DDI_ISTC_rep = np.full((N_net, N_rep, test_window, N_x, N_x), np.nan)
response_ISTC = np.full((N_net, N_rep, test_window, N_x, N_x, N_evo + 1), np.nan)
completed_istc_net = np.zeros(N_net, dtype=bool)
if os.path.exists(RESULT_PATH):
    with open(RESULT_PATH, 'rb') as f:
        previous_result = pickle.load(f)
    previous_shape = np.asarray(previous_result.get('DDI_ISTC_net', [])).shape
    if previous_shape == DDI_ISTC_net.shape:
        DDI_ISTC_net[:] = previous_result['DDI_ISTC_net']
        DDI_ISTC_rep[:] = previous_result['DDI_ISTC_rep']
        response_ISTC[:] = previous_result['response_ISTC']
        completed_istc_net[:] = previous_result['completed_istc_net']
combined_result = {'A_tensor': A_tensor, 'true_edge_mask': true_edge_mask, 'lag_list': lag_list, 'offset': offset, 'R_MS_rep': R_MS_rep, 'R_MS_net': R_MS_net, 'response_true': response_true, 'DDI_ISTC_net': DDI_ISTC_net, 'DDI_ISTC_rep': DDI_ISTC_rep, 'response_ISTC': response_ISTC, 'completed_istc_net': completed_istc_net, 'config': {**true_data['config'], 'RESERVOIR_SEED': RESERVOIR_SEED, 'N_dim_segments': N_dim_segments, 'N_units_segments': N_units_segments}}
for net in range(N_net):
    if completed_istc_net[net]:
        continue
    np.random.seed(RESERVOIR_SEED + net)
    esn = ESN(N_units_segments, N_dim_segments, N_dim_segments)
    esn.train(X_washout_segments, X_t_segments, Y_t_segments)
    for lag_idx in range(test_window):
        (DDI, DDI_rep, response) = calculate_istc_response(esn, X_test_segments, offset[net], lag_idx)
        DDI_ISTC_net[net, lag_idx] = DDI
        DDI_ISTC_rep[net, :, lag_idx] = DDI_rep
        response_ISTC[net, :, lag_idx] = response
    completed_istc_net[net] = True
    with open(RESULT_PATH, 'wb') as f:
        pickle.dump(combined_result, f)


In [ ]:
with open(RESULT_PATH, 'rb') as f:
    loaded_data = pickle.load(f)
if not np.all(loaded_data['completed_istc_net']):
    raise RuntimeError('Part 2 is incomplete; finish all ESN blocks before plotting.')
lag_list = np.asarray(loaded_data['lag_list'], dtype=int)
true_edge_mask = np.asarray(loaded_data['true_edge_mask'], dtype=bool)
R_MS_net = read_mean_squared_response(loaded_data, 'net')
DDI_ISTC_net = np.asarray(loaded_data['DDI_ISTC_net'], dtype=float)
R_MS_mean = np.mean(R_MS_net, axis=0)
DDI_ISTC_mean = np.mean(DDI_ISTC_net, axis=0)
response_selected = R_MS_mean[true_edge_mask]
istc_selected = DDI_ISTC_mean[true_edge_mask]
selected_delay = lag_list[np.nonzero(true_edge_mask)[0]]
if response_selected.size < 2 or not (np.all(np.isfinite(response_selected)) and np.all(np.isfinite(istc_selected))):
    raise ValueError('At least two finite true-edge scores are required.')
fit_denominator = np.dot(istc_selected, istc_selected)
if fit_denominator <= 0:
    raise ValueError('All ISTC scores are zero; the proportional fit is undefined.')
fit_slope = np.dot(istc_selected, response_selected) / fit_denominator
rho = float(spearmanr(response_selected, istc_selected)[0])
with plt.rc_context({'mathtext.fontset': 'stix'}):
    (fig, ax) = plt.subplots(figsize=(7.8, 6.0), facecolor='white')
    fig.subplots_adjust(left=0.14, right=0.8, bottom=0.15, top=0.88)
    scatter = ax.scatter(istc_selected, response_selected, c=selected_delay, cmap='cividis', s=65, edgecolor='white', linewidth=0.6, zorder=3)
    fit_x = np.linspace(0, np.max(istc_selected), 100)
    ax.plot(fit_x, fit_slope * fit_x, color='#333333', linewidth=1.6)
    ax.set_xlabel('DDI', fontsize=16)
    ax.set_ylabel('$R_{\\mathrm{MS}}$', fontsize=18)
    ax.set_title(f'$\\rho_s={rho:.2f}$' + '$\\qquad$' + f'$s={fit_slope:.2g}$', fontsize=14, pad=14)
    ax.set_xlim(0, 1.07 * np.max(istc_selected))
    ax.set_ylim(0, 1.07 * max(np.max(response_selected), fit_slope * np.max(istc_selected)))
    ax.xaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=4))
    ax.ticklabel_format(axis='both', style='sci', scilimits=(0, 0), useMathText=True)
    ax.tick_params(axis='both', labelsize=12, width=1.0, direction='out')
    ax.xaxis.get_offset_text().set_fontsize(10)
    ax.yaxis.get_offset_text().set_fontsize(10)
    ax.grid(False, which='both')
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
    colorbar_axes = fig.add_axes([0.84, 0.15, 0.027, 0.73])
    delay_colorbar = fig.colorbar(scatter, cax=colorbar_axes)
    delay_colorbar.set_label('Time delay', fontsize=13, labelpad=9)
    delay_colorbar.set_ticks(lag_list)
    delay_colorbar.ax.tick_params(labelsize=11, width=0.8)
    delay_colorbar.outline.set_linewidth(0.8)
    delay_colorbar.ax.grid(False, which='both')
    fig.savefig(FIGURE_PATH, dpi=600, facecolor='white')
    plt.show()
print(FIGURE_PATH)
